# Voice Changer — your voice in, bro's voice out (Google Colab)

You record yourself talking. This notebook re-skins your voice so it sounds
like **bro's voice (Atlas)** — your words, your energy, his sound.
Runs on Google's free GPU. Nothing installs on your PC. Free.

## Before you press anything — ONE setting
Top menu: **Runtime → Change runtime type → Hardware accelerator: GPU (T4)** → Save.

## How to use it
1. Run the cells **in order, top to bottom** (hover a cell, click its ▶ play button).
2. Steps 1–2 are setup (~5–10 minutes, first time only).
3. Step 3 asks you to **upload your recording** — a voice note file from your phone works.
4. Step 4 converts it. The result plays right in the notebook. **Download it** —
   Colab deletes everything when the session ends.

## Recording tips (these decide the quality)
- Quiet room, phone close to your mouth, no TV or street noise.
- Talk naturally, like you're telling a story — 20 to 60 seconds is plenty per run.
- Your recording never leaves your own Colab session.

In [ ]:
# GPU check — this must say a GPU name (usually Tesla T4)
import torch
if not torch.cuda.is_available():
    raise SystemExit("NO GPU FOUND. Go to Runtime -> Change runtime type -> GPU (T4), Save, then run this cell again.")
print("GPU OK:", torch.cuda.get_device_name(0))

## Step 1 — Install the voice changer (Seed-VC)
About 3–6 minutes. Text will scroll — that's normal.
If it ends in a red error, send bro the last few lines.

In [ ]:
%cd /content
!git clone -q https://github.com/Plachtaa/seed-vc
%cd /content/seed-vc
!pip install -q -r requirements.txt
print("Install done.")

## Step 2 — Load bro's voice (the target)
His voice sample is built into this notebook — nothing to upload.
Press play on the sample to hear exactly whose voice you'll get.

In [ ]:
import base64
from IPython.display import Audio, display

ATLAS_B64 = (
    "..."
)
with open("/content/atlas_target.mp3", "wb") as fh:
    fh.write(base64.b64decode(ATLAS_B64))
!ffmpeg -y -v error -i /content/atlas_target.mp3 -ar 22050 -ac 1 /content/atlas_target.wav
print("Target voice ready — this is bro (Atlas). Your voice will come out sounding like this:")
display(Audio("/content/atlas_target.mp3"))

## Step 3 — Upload YOUR recording
Click **Choose Files** when it appears and pick the voice file from your phone
or computer (voice notes, mp3, m4a, wav — all fine).
Listen to the playback to confirm it's you, clean and clear.

In [ ]:
from google.colab import files
from IPython.display import Audio, display

uploaded = files.upload()
name = list(uploaded.keys())[0]
with open("/content/uploaded_audio", "wb") as fh:
    fh.write(uploaded[name])
!ffmpeg -y -v error -i /content/uploaded_audio -ar 22050 -ac 1 /content/my_voice.wav
print("Got your recording:", name)
display(Audio("/content/my_voice.wav"))

## Step 4 — CONVERT
First conversion downloads the AI brain (a few hundred MB) — give it a few
extra minutes. After that, each conversion takes about 1–3 minutes.
The converted voice plays at the bottom. **Download it before you close the tab.**

Want another recording converted? Re-run Step 3 and Step 4 — nothing else.
If the pitch sounds wrong, set `AUTO_F0 = False` below and run this cell again.

In [ ]:
import subprocess, glob, os, shutil
from IPython.display import Audio, display

AUTO_F0 = True   # match your pitch level to the target voice

cmd = [
    "python", "inference.py",
    "--source", "/content/my_voice.wav",
    "--target", "/content/atlas_target.wav",
    "--output", "/content/output",
    "--diffusion-steps", "30",
    "--length-adjust", "1.0",
    "--inference-cfg-rate", "0.7",
    "--f0-condition", "False",
    "--auto-f0-adjust", str(AUTO_F0),
    "--fp16", "True",
]
result = subprocess.run(cmd, cwd="/content/seed-vc",
                        capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-1500:])
    print(result.stderr[-1500:])
    raise SystemExit("Conversion failed — send bro the lines above.")

wavs = sorted(glob.glob("/content/output/**/*.wav", recursive=True),
              key=os.path.getmtime)
if not wavs:
    raise SystemExit("No converted file appeared — send bro a screenshot of this cell.")
shutil.copy(wavs[-1], "/content/converted_voice.wav")
print("DONE — you, in bro's voice:")
display(Audio("/content/converted_voice.wav"))

## Notes
- **Zero-shot means no training** — the AI copies the target voice from the
  26-second sample alone. Good, not magic: a noisy recording in = a muddy voice out.
- Longer speech? Split it into chunks under a minute and convert each one.
- Everything runs inside **your** Colab, on **your** Google account.
  Your recordings are not sent anywhere else.
- There is a stronger route later (training a model on the target voice —
  "RVC-style") if you like this and want more quality.